# MTH 4320 / 5320 - Homework 2

## Multilayer Perceptrons, Backpropagation, and Autoencoders

**Deadline:** announced in Canvas  

**Points**: 60

MTH 5320 students earn the total points from Problems 1-6, up to 60 points.

MTH 4320 students earn 60(points from Problems 1-5)/50, plus up to 5 bonus points equal to half of the Problem 6 score.

### Instructions

- Submit **one** Jupyter notebook file and (optionally) **one** PDF with your handwritten work. Alternatively, type mathematical solutions in markdown cells in the notebook.

    - Your notebook file must include text explanations of your work, well-commented code, and the outputs from your code.

    - All mathematical work must be shown for written/typed problems.

    - All neural-network implementations must use raw NumPy. Do not use PyTorch, Keras, or any other autograd library. You may use scikit-learn utilities for any other tasks.

## Problems

### Problem 1 - MLP Architecture and Forward Pass (10 pts)

Consider $x=\begin{pmatrix}1&2\end{pmatrix}$, one-hot target $y=\begin{pmatrix}0&1\end{pmatrix}$, and

$$
W^{(1)}=\begin{pmatrix}1&-1\\0.5&1\end{pmatrix},
\quad b^{(1)}=\begin{pmatrix}0.5&-0.5\end{pmatrix},
\quad W^{(2)}=\begin{pmatrix}1&-1\\-2&2\end{pmatrix},
\quad b^{(2)}=\begin{pmatrix}0.25&-0.25\end{pmatrix}.
$$

Use sigmoid in the hidden layer and softmax in the output layer.

**(i)** Draw and label the network.

**(ii)** State the dimensions of all inputs, parameters, pre-activations, and activations.

**(iii)** Calculate the full forward pass and categorical cross-entropy loss by hand. Show intermediate values and round to at least four decimal places.

**(iv)** Remove the hidden sigmoid and show algebraically that the two affine transformations reduce to one affine transformation for the output logits. Explain why MLPs need nonlinear hidden activations.

### Problem 2 - Verify Backpropagation with Numerical Gradients (15 pts)

Use centered finite differences to check the Week 4 `MultilayerPerceptron` gradients:

$$
\frac{\partial L}{\partial\theta}\approx\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}.
$$

For each entry, calculate relative error as

$$
\frac{|g_{\mathrm{analytical}}-g_{\mathrm{numerical}}|}{\max\!\left(10^{-12},|g_{\mathrm{analytical}}|+|g_{\mathrm{numerical}}|\right)}.
$$

**(i)** Check every weight and bias entry for one small `loss="cce"` network using a synthetic batch. Restore each parameter after perturbing it and do not update the model.

**(ii)** For each parameter array, report its shape and the maximum absolute and relative differences between numerical and analytical gradients.

**(iii)** Test several substantially different positive values of $\varepsilon$. Explain the errors caused by values that are too large or too small.

**(iv)** Remove the hidden activation derivative from a copy of backpropagation and show that the checker detects the error. Explain why decreasing training loss alone is a weaker test.

### Problem 3 - Replace Sigmoid with ReLU (10 pts)

Extend `StochasticMultilayerPerceptron` to use ReLU rather than sigmoid in its hidden layers. Preserve both output modes: softmax for `loss="cce"` and linear for `loss="mse"`.

**(i)** Modify both passes, cache the required values, and state your derivative at zero.

**(ii)** Explain why replacing only the forward activation is incorrect and why ReLU needs different cached information than sigmoid.

**(iii)** Demonstrate that a negative hidden pre-activation blocks that example's gradient through the unit.

**(iv)** Train matched sigmoid and ReLU networks on `sklearn.datasets.make_moons()`. Convert its labels to two-column one-hot targets. Keep the split, architecture, seed, learning rate, batch size, regularization, and epochs fixed. Compare loss and accuracy.

### Required MNIST Data for Problems 4-6

Run the following cell without modifying its split, corruption probability, or seeds. Problems 4 and 5 use only the clean `X_train`, `X_val`, and `X_test` arrays. The three `X_*_corrupted` arrays are created exclusively for Problem 6.

In [ ]:
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


def salt_and_pepper_noise(X, probability, rng):
    random_values = rng.random(X.shape, dtype=np.float32)
    X_corrupted = X.copy()
    X_corrupted[random_values < probability / 2] = 0.0
    X_corrupted[random_values > 1 - probability / 2] = 1.0
    return X_corrupted


X, y = fetch_openml(
    "mnist_784", version=1, return_X_y=True, as_frame=False
)
X = X.astype(np.float32) / 255.0

X_train_val, X_test, y_train_val, _ = train_test_split(
    X, y, test_size=10_000, random_state=1, stratify=y
)
X_train, X_val = train_test_split(
    X_train_val,
    test_size=10_000,
    random_state=2,
    stratify=y_train_val,
)

# Labels are used only to make balanced splits.
del X, y, X_train_val, y_train_val

# The corrupted arrays below are used only in Problem 6.
rng = np.random.default_rng(3)
X_train_corrupted = salt_and_pepper_noise(X_train, 0.30, rng)
X_val_corrupted = salt_and_pepper_noise(X_val, 0.30, rng)
X_test_corrupted = salt_and_pepper_noise(X_test, 0.30, rng)


### Problem 4 - Build an MNIST Autoencoder (10 pts)

Use the ReLU `StochasticMultilayerPerceptron` from Problem 3 as a fully connected autoencoder for flattened, scaled $28\times28$ MNIST images. Select `loss="mse"` and train with each image as both input and target. Use

$$
784\longrightarrow h\longrightarrow k\longrightarrow h\longrightarrow784,
$$

where $k<784$ is the bottleneck. Use ReLU in every hidden layer, including the bottleneck. The existing MSE mode supplies the linear reconstruction output, so reuse the class rather than create an unrelated implementation.

**(i)** Describe the encoder, bottleneck, and decoder and state all parameter dimensions.

**(ii)** Explain how selecting MSE changes the output activation, loss, and output-layer error compared with CCE classification.

**(iii)** Train with shuffled mini-batches and plot training and validation reconstruction loss.

**(iv)** Display at least five validation images beside their reconstructions.

**(v)** Explain how the bottleneck restricts the network's ability to copy all 784 inputs directly.

### Problem 5 - Investigate the Bottleneck (5 pts)

**(i)** Compare at least three substantially different bottleneck dimensions using a fixed 10,000-example subset of the training split. Keep the subset, initialization seed, training settings, and evaluation procedure fixed. Train only the selected model on the full training split.

**(ii)** Report dimension retained, parameter count, training and validation loss, training time, and matched reconstruction examples. Plot validation loss against bottleneck dimension.

**(iii)** Select the smallest bottleneck whose validation MSE is within 10% of the lowest validation MSE in your comparison. Use matched reconstructions as a secondary check. Discuss what is lost as the bottleneck shrinks and distinguish dimensional from file-size compression.

### Problem 6 - Sparse Denoising Autoencoder

**10 pts; required for graduate students, optional 5-point bonus for undergraduate students**

Extend the autoencoder into a sparse denoising autoencoder with a sigmoid bottleneck.

A **denoising autoencoder** receives a noisy image but is trained to reconstruct the corresponding clean image. Its goal is to learn stable structure in the data rather than simply copy each input value, making its representation and reconstruction more resistant to noise.

A **sparse autoencoder** encourages most bottleneck units to remain inactive for any given input. Its goal is to represent an image using a small, selective set of learned features.

In this problem, you will study these ideas separately and together by comparing ordinary, sparse, denoising, and sparse denoising autoencoders. This four-model comparison will help you distinguish the effects of input corruption, bottleneck sparsity, and their interaction.

#### A. Denoising

Use the required MNIST split and corruption arrays created before Problem 4. Train only the four final comparison models on the full training split; tune $\beta$ on the fixed 10,000-example search subset.

Train with `X_train_corrupted` as input and `X_train` as target. Validate against `X_val_corrupted` and `X_val`. Show example clean/corrupted pairs and explain the denoising objective.

#### B. Sparsity

For bottleneck activation $a_{ij}$, define $\widehat\rho_j=B^{-1}\sum_i a_{ij}$. Use $\rho=0.05$ and add

$$
\beta\sum_{j=1}^{k}\left[\rho\log\frac{\rho}{\widehat\rho_j}+(1-\rho)\log\frac{1-\rho}{1-\widehat\rho_j}\right]
$$

to the reconstruction loss. Derive its derivative with respect to $\widehat\rho_j$ and its contribution to the bottleneck error. Clip $\widehat\rho_j$ to $[10^{-12},1-10^{-12}]$ before evaluating logarithms or divisions. Implement the penalty and record reconstruction loss, sparsity penalty, total loss, and average bottleneck activations.

#### C. Comparison

Compare four otherwise matched MSE autoencoders with sigmoid bottlenecks: a standard autoencoder trained on clean inputs with $\beta=0$; a sparse autoencoder trained on clean inputs; a denoising autoencoder trained on corrupted inputs with $\beta=0$; and a sparse denoising autoencoder trained on corrupted inputs. Use validation data to select $\beta$, then use that same selected value for both sparse models. Evaluate every model against clean targets twice: once with clean inputs and once with the fixed $p=0.30$ corrupted inputs. Report both reconstruction losses, average bottleneck activation, fraction of activations near zero, and matched reconstructions. Include a bottleneck-activation heatmap and discuss the individual and combined effects of denoising and sparsity.